# AirBnB NYC Analytics Project 

* Project by Nhi Bui · Villanova University · [GitHub](https://github.com/nhibui23/airbnb-saas-product-analytics) · [LinkedIn](https://linkedin.com/in/nhiuyenbui)

## 03. Statistical Testing: What Actually Drives Review Ratings?

> "Of all the factors a host controls, which ones best impact ratings?"

This notebook tests whether those patterns are statistically significant or just noise using ANOVA with Tukey's HSD post-hoc correction. We evaluate the 4 host-controlled factors: availability, minimum nights, price, and host identity verification.

**Key findings**
- Which factors have a statistically significant effect on review ratings (p < 0.05)
- The relative strength of each factor
- Which factors product can ignore vs. invest in

# Statistical Testing for Factor Impact on Review Ratings

## Question 1
What host listing factors controlled by hosts most significantly impact review ratings?

## My proposed approach
Validate EDA findings with statistical tests to determine 
if the observed differences in review ratings across host-controlled 
factors are statistically significant.

In [89]:
#Reload data
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/Airbnb_Open_Data_Cleaned.csv')

# ANOVA Test - Room Type

In [90]:
#Split data into groups by room type
df['room type'].unique()

<StringArray>
['Private room', 'Entire home/apt', 'Shared room', 'Hotel room']
Length: 4, dtype: str

In [91]:
# Create a variable for each group's review ratings
private = df[df['room type'] == 'Private room']['review rate number']
entire = df[df['room type'] == 'Entire home/apt']['review rate number']
shared = df[df['room type'] == 'Shared room']['review rate number']
hotel = df[df['room type'] == 'Hotel room']['review rate number']

In [92]:
#Run ANOVA test to compare the means of the review ratings across the different room types
f_stat, p_value = stats.f_oneway(private, entire, shared, hotel)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 3.2804
P-value: 0.0200


Because p-value = 0.02 < 0.05

=> There is a statistically significant difference in review ratings between room types

=> Reject the NULL hypothesis

=> This confirms our earlier statement of review rate number affected by different room types

## Post-hoc test

To find out which specific room types differ from each other, you need a post-hoc test using Tukey's HSD 

In [93]:
#Check for Turkey HSD post-hoc test to see which groups differ
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df['review rate number'], df['room type'], alpha=0.05)
print(tukey)

       Multiple Comparison of Means - Tukey HSD, FWER=0.05        
     group1        group2    meandiff p-adj   lower  upper  reject
------------------------------------------------------------------
Entire home/apt   Hotel room    0.265 0.1932 -0.0778 0.6079  False
Entire home/apt Private room    0.025 0.0724 -0.0015 0.0514  False
Entire home/apt  Shared room   0.0321  0.801 -0.0588 0.1229  False
     Hotel room Private room  -0.2401 0.2739  -0.583 0.1028  False
     Hotel room  Shared room   -0.233  0.328 -0.5867 0.1208  False
   Private room  Shared room   0.0071 0.9971 -0.0841 0.0983  False
------------------------------------------------------------------


### Key Takeaway

- **ANOVA finds a real difference across room types** (p = 0.02), but Tukey's test shows **no individual pair stands out** as significantly different
- The biggest contrast is Entire home/apt vs Private room (p = 0.0724) 
- The effect is **real but small**, as it spread across all groups instead of driven by one room type

### Interpretation

- ANOVA tells us that something varies between room types, but Tukey's test shows no 2 room types are different enough
- This usually happens whenevery room type adds a little to the gap, but no 2 stand far enough apart to single out
- So the EDA finding that "Hotel rooms lead and Entire homes lag" is in the right direction, but not strong enough for Airbnb to base a strategy on

### Recommendation

1. The evidence isn't strong enough to recommend Entire home hosts switch to Hotel-room formats

2. Airbnb should focus on quality features that work for all room types:
* **Amenity checklist** 

Adjust amenties based on type of home. (Ex: entire home hosts see whole-apartment essentials, while private room hosts value shared-living essentials)

* **Response time benchmarks** 

Hosts can see how their communication stacks up against top performers in their category

In [94]:
#Sample sizes
df['room type'].value_counts()

room type
Entire home/apt    34100
Private room       28167
Shared room         1359
Hotel room            92
Name: count, dtype: int64

However, hotel room's highest rating (3.54) should be interpreted with 
caution due to its  smaller sample size (92 listings) compared to 
other room types (1300-34000 listings)

# ANOVA Test - Price

In [95]:
#Check bins of price
df['price'].dtype

dtype('float64')

Because type of price is float, we need to recreate the bins

In [96]:
#Recreate bins for price
df['price_group'] = pd.cut(df['price'], 
                           bins=[0, 50, 500, 1000, 1500], 
                           labels=['Low', 'Medium', 'High', 'Very High'])

In [97]:
# Create a variable for each group's review ratings
low = df[df['price_group'] == 'Low']['review rate number']
medium = df[df['price_group'] == 'Medium']['review rate number']
high = df[df['price_group'] == 'High']['review rate number']
very_high = df[df['price_group'] == 'Very High']['review rate number']

In [98]:
#Run ANOVA test to compare the means of the review ratings across the different price groups
f_stat, p_value = stats.f_oneway(low, medium, high, very_high)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 5.5889
P-value: 0.0008


Because p-value = 0.0008 < 0.05

=> There is a statistically significant difference in review ratings between prices

=> Reject the NULL hypothesis

=> This confirms our earlier statement of review rate number affected by different prices

## Post-hoc test

To find out which specific price levels differ from each other, you need a post-hoc test using Tukey's HSD 

In [99]:
#Check for Turkey HSD post-hoc test to see which groups differ
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df['review rate number'], df['price_group'], alpha=0.05)
print(tukey)

  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1   group2  meandiff p-adj   lower   upper  reject
-------------------------------------------------------
  High       Low  -0.1429 0.7972 -0.5446  0.2587  False
  High    Medium   0.0153 0.5159 -0.0134   0.044  False
  High Very High  -0.0427 0.0153 -0.0796 -0.0059   True
   Low    Medium   0.1583 0.7422 -0.2434    0.56  False
   Low Very High   0.1002 0.9191 -0.3022  0.5026  False
Medium Very High  -0.0581 0.0004 -0.0955 -0.0206   True
-------------------------------------------------------


### Key Takeaway

* **2 pairs are statistically significant:**
  - High vs Very High (p = 0.0153): significant difference
  - Medium vs Very High (p = 0.0004):highly significant
* Both involve **Very High price listings rating lower** than mid and high-tier listings
* **Budget vs everything else came back not significant** 

### Interpretation

* This confirms our hypothesis from EDA: guests paying Very High prices expect a lot, and when listings don't deliver, ratings suffer
* The mid-tier and high-tier listings sit in the sweet spot, as pricing is high enough to signal quality, but not so high that guests feel let down
* The budget tier looked weak in EDA but doesn't hold up under testing 

### Recommendation

1. The data shows that pushing into the Very High tier without matching quality is a setup for disappointment. Airbnb thereby should make sure hosts pricing in that tier are actually equipped to deliver on it.

2. 3 features that work together:
* **Pricing quality check** 

Alert  hosts when their price is much higher than the neighborhood average without matching amenities or photo quality


* **Premium listing requirements** 

For Very High priced listings, we require more detailed descriptions, verified amenity lists, and professional photos before the listing goes live


* **Pricing recommendation tool**

Airbnb should suggest a price range that optimizes for both revenue and guest satisfaction

Given the smaller effect sizes observed in EDA (0.08 points), statistical testing may not needed for availability and minimum nights as they are unlikely to reach significance

However, for completion and assurance of risks check, we will still conduct tests for these 2 factors

In [100]:
#Sample sizes
df['price_group'].value_counts()

price_group
High         27684
Medium       24823
Very High    11144
Low             67
Name: count, dtype: int64

Similarly, low price group should also be interpreted with 
caution due to its smaller sample size (67) compared to 
other price levels (10000-27000 listings)

# ANOVA Test - Availability

In [101]:
#Check bins for availability
df['availability 365'].dtype

dtype('float64')

In [102]:
#Recreate bins for availability
df['availability_group'] = pd.cut(df['availability 365'], 
                                   bins=[0, 90, 180, 270, 365], 
                                   labels=['Low', 'Medium', 'High', 'Always Available'])

In [103]:
# Create a variable for each group's review ratings
a_low = df[df['availability_group'] == 'Low']['review rate number']
a_medium = df[df['availability_group'] == 'Medium']['review rate number']
a_high = df[df['availability_group'] == 'High']['review rate number']
a_very_high = df[df['availability_group'] == 'Always Available']['review rate number']

In [104]:
#Run ANOVA test to compare the means of the review ratings across the different availability groups
f_stat, p_value = stats.f_oneway(a_low, a_medium, a_high, a_very_high)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 8.6482
P-value: 0.0000


Since p < 0.00001, this is the most significant result across all factors

=> Despite the small 0.08 gap in EDA, with 63K+ data points even small differences can be statistically significant 
=> Need to test Turkey HSD to actually test this difference

## Post-hoc test

In [105]:
#Check the Turkey HSD post-hoc test to see which groups differ
tukey_avail = pairwise_tukeyhsd(df['review rate number'], df['availability_group'], alpha=0.05)
print(tukey_avail)

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
     group1      group2 meandiff p-adj   lower   upper  reject
--------------------------------------------------------------
Always Available   High  -0.0493   0.01 -0.0899 -0.0086   True
Always Available    Low   0.0263 0.1649 -0.0064  0.0591  False
Always Available Medium  -0.0127 0.8234 -0.0506  0.0251  False
            High    Low   0.0756    0.0  0.0364  0.1148   True
            High Medium   0.0365 0.1364  -0.007  0.0801  False
             Low Medium  -0.0391 0.0291 -0.0753 -0.0028   True
--------------------------------------------------------------


### Key Takeaway

* **Tukey's HSD identifies 3 significant pairs:**
  * High vs Low (p = 0.00): Low availability rates significantly higher
  * Always Available vs High (p = 0.01) 
  * Low vs Medium (p = 0.03)
* **Availability is the strongest predictor** of review ratings out of all factors tested
* **High availability (181-270 days) consistently underperforms**, which is also the worst-performing tier across comparisons

### Interpretation

* The finding confirms our EDA hypothesis: hosts who limit availability are more selective and invest more in each guest experience
* The High tier (181-270 days) is the consistent loser because it's the worst of both worlds: enough bookings to create wear and tear, but not enough time to maintain the listing
* Surprisinly, Always Available (271-365 days) outperforms High because hosts who go year-round are usually operators with cleaning teams in place, while the High tier does not have that support

### Recommendation

1. Reframe availability from a volume lever to a quality lever. Airbnb should give hosts permission to take breaks without being penalized.

2. Product features

* **Maintenance Mode** 

As suggested previously, Airbnb should allows hosts to block 3-7 days per month for cleaning without it counting against their host responsiveness score

* **Quality Rest reminders**

Airbnb should thereby suggest a maintenance window when a listing has been continuously booked for X weeks

* **Quality intervention alerts**

Airbnb should track to warn listings with high availability and declining ratings, and provide proactive outreach from host support before ratings drop further

In [106]:
#Sample sizes
df['availability_group'].value_counts()

availability_group
Low                 22593
Always Available    18100
Medium              12839
High                10186
Name: count, dtype: int64

The sample sizes of availablity group don't have much difference compared to price and room types, so we can trust our analysis

# ANOVA Test - Minimum nights

In [107]:
#Check bins for minimum nights
df['minimum nights'].dtype

dtype('float64')

In [108]:
#Because we can't run the bins, we have to check category again
df['minimum nights'].head()

0    10.0
1    30.0
2    10.0
3    45.0
4     2.0
Name: minimum nights, dtype: float64

In [109]:
#Create bins for minimum nights
df['min_nights'] = pd.cut(df['minimum nights'], 
                                   bins=[1, 3, 8, 30, 2645], 
                                   labels=['Short stay', 'Medium stay', 'Monthly stay', 'Long term'])

In [110]:
#Create variable for each group's review ratings
m_short = df[df['min_nights'] == 'Short stay']['review rate number']
m_medium = df[df['min_nights'] == 'Medium stay']['review rate number']
m_monthly = df[df['min_nights'] == 'Monthly stay']['review rate number']
m_long_term = df[df['min_nights'] == 'Long term']['review rate number']


In [111]:
#Run ANOVA test to compare the means of the review ratings across the different room types
f_stat, p_value = stats.f_oneway(m_short, m_medium, m_monthly, m_long_term)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 8.1367
P-value: 0.0000


Similarly, since p < 0.00001, this is also the most significant result across all factors

=> Need to test Turkey HSD to actually test this difference

In [ ]:
#Drop NaN values for minimum nights
df_nights = df.dropna(subset=['min_nights'])

In [115]:
#Run Turkey HSD post-hoc test to see which groups differ
tukey_nights = pairwise_tukeyhsd(df_nights['review rate number'], df_nights['min_nights'], alpha=0.05)
print(tukey_nights)

      Multiple Comparison of Means - Tukey HSD, FWER=0.05       
   group1       group2    meandiff p-adj   lower   upper  reject
----------------------------------------------------------------
   Long term  Medium stay   0.0039 0.9997 -0.1066  0.1144  False
   Long term Monthly stay   0.0815 0.2315 -0.0292  0.1922  False
   Long term   Short stay    0.016 0.9812 -0.0918  0.1238  False
 Medium stay Monthly stay   0.0776 0.0001  0.0322  0.1231   True
 Medium stay   Short stay   0.0121 0.8444 -0.0258    0.05  False
Monthly stay   Short stay  -0.0655 0.0001  -0.104 -0.0271   True
----------------------------------------------------------------


### Key Takeaway

* **Tukey's HSD identifies 2 significant pairs:**
  * Medium stay vs Monthly stay (p = 0.0001)
  * Monthly stay vs Short stay (p = 0.0001) 
* **Monthly stays (8-30 nights) clearly outperform** both short-term and medium-term stays
* The effect is strong, as both pairs are highly significant at p = 0.0001

### Interpretation

* Monthly guests are **more invested in the space**, as they've planned the trip, settled in, and have realistic expectations
* Hosts may also **screen more carefully** for longer commitments
* The host-guest relationship benefits from **extended interaction**, as small issues get communicated and resolved
* The finding aligns with the EDA observation and pairs naturally with the availability finding: hosts who run monthly stays often have low availability, and both factors point toward the same quality strategy

### Recommendation

1. **Monthly Stay tier** 

This is dedicated listing category with its own search filter, elevated placement, and host badges to signal commitment-friendly listings

2. **Monthly stay discount tools** 

Airbnb should have built-in pricing logic that auto-suggests discount tiers (e.g., 10% off for 1+ week, 25% off for 1+ month) to attract longer-term guests

3. **Monthly Stay Specialist badge** 

Similarly, offering badges for hosts who consistently deliver high-rated stays will give them a reason to keep optimizing for the segment

Finally, we test the last remaining factor, host_identity_verified, which showed the smallest difference in EDA (0.001 points)

Given the negligible gap observed, we expect this to show no statistical significance, but also test for completeness.


In [85]:
#Sample sizes
df['min_nights'].value_counts()

min_nights
Short stay      25697
Medium stay     10690
Monthly stay    10252
Long term         966
Name: count, dtype: int64

Similarly, long term night group should also be interpreted with 
caution due to its smaller sample size (966) compared to 
other minimum nights levels (10000-25000)

# Host-identity-verified

In [116]:
#Run t-test to compare the means of the review ratings between hosts that verifiy and those that do not verify their identity
verified = df[df['host_identity_verified'] == 'verified']['review rate number']
unconfirmed = df[df['host_identity_verified'] == 'unconfirmed']['review rate number']

t_stat, p_value = stats.ttest_ind(verified, unconfirmed, equal_var=False)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")


T-statistic: 0.1113
P-value: 0.9114


### Key Takeaway

With p = 0.62 > 0.05, we can say host identity verification has no meaningful impact on review ratings

=> This confirms our earlier EDA finding and suggests Airbnb should not prioritize verification status as a quality indicator

# Overall Conclusion

After conducting formal hypothesis testing across all six host-controlled factors, we land at 3 key insights:

**1. Availability is the strongest driver of review ratings.**

Contrast to our previous EDA finding, hosts with lower availability achieve significantly higher ratings, confirming that selective hosting and investing in quality over volume leads to better guest experiences. 

=> Airbnb should encourage hosts to schedule maintenance breaks rather than listing year-round.

**2. Monthly stays hit the sweet spot for guest satisfaction.**

Similarly, listings with 8-30 night minimum stays significantly outperform short and medium stays. The extended host-guest relationship and higher commitment from both parties drives better experiences. 

=> Airbnb should promote monthly stays as a premium tier.

**3. Very high pricing creates an expectation gap that hurts ratings.**

As expected from EDA testing, very-high-priced listings ($1000+) rate significantly lower than medium and high-priced listings. Guests paying top dollar expect superior experiences, and when reality falls short, ratings suffer. 

=> Airbnb should provide pricing guidance tools that help hosts find their optimal price-to-quality ratio.

* Meanwhile, Host Verification showed no statistically significant impact on review ratings.

=> This suggests Airbnb should redirect product development resources away from these features and toward the the 3 high-impact areas identified above.